# Benchmark Results Visualization

**Part A — CNN architecture benchmark:** Compare channel × scaler × CNN-arch combinations.  
**Part B — CNN vs FreeSurfer ROI:** Head-to-head comparison of untrained CNN features against classical ROI morphometrics.

In [ ]:
import re
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

RESULTS_CSV = 'outputs/results/evaluation_results.csv'

HIGHER_IS_BETTER = {'balanced_accuracy', 'auc', 'r2', 'pearson', 'spearman', 'f1_macro'}
LOWER_IS_BETTER  = {'mae'}

CNN_CHANNEL_ORDER  = ['t1', 't1_sobel', 't1_rank_sobel', 't1_median_sobel']
CNN_SCALER_ORDER   = ['zscore', 'robust', 'minmax']
ROI_CHANNEL_ORDER  = ['aseg_vol', 'aparc_thick', 'aparc_vol', 'aparc_area', 'aseg_aparc']
ROI_SCALER_ORDER   = ['none', 'icv']
CNN_ARCH_ORDER     = ['double_conv', 'cov_pool', 'freesurfer_roi']

ARCH_PALETTE = {
    'double_conv':    '#4C72B0',
    'cov_pool':       '#DD8452',
    'freesurfer_roi': '#55A868',
}
CNN_CHANNEL_PALETTE = dict(zip(CNN_CHANNEL_ORDER, sns.color_palette('tab10', 4)))
ROI_CHANNEL_PALETTE = dict(zip(ROI_CHANNEL_ORDER, sns.color_palette('Set2', 5)))

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

# ---------------------------------------------------------------------------
# Load & enrich results
# ---------------------------------------------------------------------------

def infer_cnn_arch(feature_file: str) -> str:
    """Extract cnn_arch from the feature filename for pre-cnn_arch-column results."""
    m = re.search(r'model-([^_]+)', str(feature_file))
    return m.group(1) if m else 'double_conv'


raw = pd.read_csv(RESULTS_CSV)
# Backward-compat: old model keys lacked task-type suffix
raw['model'] = raw['model'].replace({'ridge': 'ridge_reg', 'logistic': 'logistic_clf'})

if 'cnn_arch' not in raw.columns:
    raw['cnn_arch'] = raw['feature_file'].apply(infer_cnn_arch)
else:
    raw['cnn_arch'] = raw['cnn_arch'].fillna('double_conv')

# Convenience splits
cnn_res = raw[raw['cnn_arch'] != 'freesurfer_roi'].copy()
roi_res = raw[raw['cnn_arch'] == 'freesurfer_roi'].copy()

# Ordered categoricals for CNN subset
cnn_res['channels'] = pd.Categorical(cnn_res['channels'], categories=CNN_CHANNEL_ORDER, ordered=True)
cnn_res['scaler']   = pd.Categorical(cnn_res['scaler'],   categories=CNN_SCALER_ORDER,  ordered=True)

# Ordered categoricals for ROI subset
if not roi_res.empty:
    roi_res['channels'] = pd.Categorical(roi_res['channels'], categories=ROI_CHANNEL_ORDER, ordered=True)
    roi_res['scaler']   = pd.Categorical(roi_res['scaler'],   categories=ROI_SCALER_ORDER,  ordered=True)

print(f'Total result rows : {len(raw):,}')
print(f'CNN rows          : {len(cnn_res):,}   archs: {sorted(cnn_res["cnn_arch"].unique())}')
print(f'ROI rows          : {len(roi_res):,}')
print(f'Tasks             : {sorted(raw["task"].unique())}')
print(f'Metrics           : {sorted(raw["metric"].unique())}')

if roi_res.empty:
    print('\n[NOTE] No FreeSurfer ROI results found. Run:')
    print('  python scripts/run_all_evaluations.py --feature-glob "features__model-freesurfer_roi*.npz"')
    print('  python scripts/summarize_results.py --results outputs/results/evaluation_results.csv')

In [ ]:
# ---------------------------------------------------------------------------
# Shared summary builder (groups by cnn_arch too)
# ---------------------------------------------------------------------------

GROUP_COLS = ['cnn_arch', 'scaler', 'channels', 'task', 'analysis_type', 'pair_mode', 'model', 'metric']

def build_summary(df: pd.DataFrame) -> pd.DataFrame:
    """Aggregate mean/std/median/q25/q75/n per group, add rank within each task slice."""
    summary = (
        df.groupby(GROUP_COLS, dropna=False)['value']
        .agg(mean='mean', std='std', median='median',
             q25=lambda s: s.quantile(0.25), q75=lambda s: s.quantile(0.75), n='count')
        .reset_index()
    )
    rank_group = ['task', 'analysis_type', 'pair_mode', 'model', 'metric']
    summary['rank'] = pd.NA
    for _, idx in summary.groupby(rank_group, dropna=False).groups.items():
        metric = summary.loc[list(idx), 'metric'].iloc[0]
        if metric not in HIGHER_IS_BETTER and metric not in LOWER_IS_BETTER:
            continue
        asc = metric in LOWER_IS_BETTER
        summary.loc[list(idx), 'rank'] = (
            summary.loc[list(idx), 'mean'].rank(method='min', ascending=asc).astype('Int64')
        )
    return summary


summary     = build_summary(raw)
cnn_summary = build_summary(cnn_res)
roi_summary = build_summary(roi_res) if not roi_res.empty else pd.DataFrame()

print(f'Summary rows (all)  : {len(summary):,}')
print(f'Summary rows (CNN)  : {len(cnn_summary):,}')
print(f'Summary rows (ROI)  : {len(roi_summary):,}')

---
# Part A — CNN Architecture Benchmark

## A1. Age regression (cross-sectional)

Boxes span Q25–Q75 across CNN seeds × split seeds. One column per scaler; hue = downstream model.  
If multiple CNN architectures are present (`double_conv`, `cov_pool`), each gets its own row block.

In [ ]:
def plot_cnn_task(cnn_res, task, analysis_type='crosssectional', pair_mode=None,
                  metrics=None, title=None, figsize_per_arch=(14, 10)):
    archs = [a for a in CNN_ARCH_ORDER if a in cnn_res['cnn_arch'].unique()]
    if not archs:
        print('No CNN results.')
        return

    mask = (cnn_res['task'] == task) & (cnn_res['analysis_type'] == analysis_type)
    if pair_mode:
        mask &= cnn_res['pair_mode'].fillna('') == pair_mode
    df = cnn_res[mask]

    default_metrics = (
        ['mae', 'r2', 'pearson', 'spearman']
        if task in ('age', 'delta_age') else
        ['balanced_accuracy', 'auc', 'f1_macro']
    )
    if metrics is None:
        metrics = [m for m in default_metrics if m in df['metric'].unique()]

    for arch in archs:
        adf = df[df['cnn_arch'] == arch]
        fig, axes = plt.subplots(len(metrics), 3,
                                  figsize=figsize_per_arch,
                                  sharey='row', sharex='col')
        if len(metrics) == 1:
            axes = axes[np.newaxis, :]

        for row, metric in enumerate(metrics):
            mdf = adf[adf['metric'] == metric]
            for col, scaler in enumerate(CNN_SCALER_ORDER):
                ax = axes[row, col]
                sdf = mdf[mdf['scaler'] == scaler]
                if sdf.empty:
                    ax.set_visible(False)
                    continue
                sns.boxplot(
                    data=sdf, x='channels', y='value', hue='model',
                    order=CNN_CHANNEL_ORDER, ax=ax,
                    flierprops=dict(marker='.', markersize=2, alpha=0.4),
                    linewidth=0.8,
                )
                ax.set_xlabel('')
                ax.set_ylabel(metric if col == 0 else '')
                ax.set_title(f'scaler={scaler}' if row == 0 else '')
                ax.tick_params(axis='x', rotation=30)
                if row == 0 and col == 2:
                    ax.legend(title='model', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
                elif ax.get_legend():
                    ax.get_legend().remove()

        pm_label = f' | pair_mode={pair_mode}' if pair_mode else ''
        fig.suptitle(f'{title or task} — arch={arch}{pm_label}', fontsize=12, y=1.01)
        fig.tight_layout()
        plt.show()


plot_cnn_task(cnn_res, 'age', title='Age regression — cross-sectional')

## A2. Classification tasks (cross-sectional)

In [ ]:
for task in ['sex', 'adult_vs_child']:
    if task in cnn_res['task'].unique():
        plot_cnn_task(cnn_res, task, figsize_per_arch=(14, 7))

## A3. Longitudinal — delta_age regression

In [ ]:
for pm in ['delta', 'concat_delta', 'concat', 'annualized_delta']:
    if not cnn_res[(cnn_res['task'] == 'delta_age') & (cnn_res['pair_mode'].fillna('') == pm)].empty:
        plot_cnn_task(cnn_res, 'delta_age', analysis_type='longitudinal',
                      pair_mode=pm, title=f'delta_age — longitudinal')

## A4. Performance heatmap — all tasks, best downstream model

Rows = (task, metric). Columns = (cnn_arch × scaler × channels). Green = best in row.

In [ ]:
METRIC_MODEL = {
    'age':          {'pearson': 'ridge_reg', 'mae': 'ridge_reg'},
    'delta_age':    {'pearson': 'ridge_reg', 'mae': 'ridge_reg'},
    'sex':          {'auc': 'logistic_clf', 'balanced_accuracy': 'logistic_clf'},
    'session_type': {'balanced_accuracy': 'rf'},
    'studies':      {'balanced_accuracy': 'rf'},
    'study_num':    {'balanced_accuracy': 'rf'},
}


def build_heatmap_data(summary, metric_model=METRIC_MODEL):
    rows = []
    for task, mmap in metric_model.items():
        for metric, prefer_model in mmap.items():
            mask = (
                (summary['task'] == task) &
                (summary['metric'] == metric) &
                (summary['model'] == prefer_model)
            )
            sub = summary[mask]
            if sub.empty:
                continue
            # For longitudinal: pick best pair_mode
            if sub['pair_mode'].notna().any():
                pm_means = sub.groupby('pair_mode')['mean'].mean()
                best_pm = pm_means.idxmin() if metric in LOWER_IS_BETTER else pm_means.idxmax()
                sub = sub[sub['pair_mode'] == best_pm]
            for _, r in sub.iterrows():
                rows.append({
                    'task_metric': f"{task}\n{metric}",
                    'config': f"{r['cnn_arch']}\n{r['scaler']}\n{r['channels']}",
                    'mean': r['mean'],
                    'metric': metric,
                })
    return pd.DataFrame(rows)


def plot_heatmap(summary, title='Performance heatmap (row-normalized, green=best)',
                 cell_fontsize=6):
    hdf = build_heatmap_data(summary)
    if hdf.empty:
        print('No data for heatmap.')
        return
    pivot = hdf.pivot_table(index='task_metric', columns='config', values='mean', aggfunc='mean')

    pivot_norm = pivot.copy()
    for idx in pivot.index:
        metric_name = idx.split('\n')[1] if '\n' in idx else ''
        row_vals = pivot.loc[idx].dropna()
        vmin, vmax = row_vals.min(), row_vals.max()
        if vmax == vmin:
            pivot_norm.loc[idx] = 0.5
        elif metric_name in LOWER_IS_BETTER:
            pivot_norm.loc[idx] = 1 - (pivot.loc[idx] - vmin) / (vmax - vmin)
        else:
            pivot_norm.loc[idx] = (pivot.loc[idx] - vmin) / (vmax - vmin)

    ncols = len(pivot.columns)
    fig, ax = plt.subplots(figsize=(max(10, ncols * 0.8), max(6, len(pivot) * 0.55)))
    im = ax.imshow(pivot_norm.values, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)

    ax.set_xticks(range(ncols))
    ax.set_xticklabels(pivot.columns, rotation=45, ha='right', fontsize=7)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index, fontsize=8)

    for i in range(len(pivot.index)):
        for j in range(ncols):
            val = pivot.iloc[i, j]
            if pd.notna(val):
                nv = pivot_norm.iloc[i, j]
                color = 'black' if 0.2 < nv < 0.8 else 'white'
                ax.text(j, i, f'{val:.3f}', ha='center', va='center',
                        fontsize=cell_fontsize, color=color)

    plt.colorbar(im, ax=ax, label='row-normalized (1=best)', fraction=0.02, pad=0.01)
    ax.set_title(title, pad=10)
    fig.tight_layout()
    plt.show()


plot_heatmap(cnn_summary, title='CNN benchmark — performance heatmap')

## A5. CNN-seed variability

Distribution over CNN seeds (mean across split seeds). One violin per channel × scaler.

In [ ]:
def plot_seed_variability(cnn_res, task, metric, model, analysis_type='crosssectional',
                          pair_mode=None, figsize=(14, 5)):
    archs = [a for a in CNN_ARCH_ORDER if a in cnn_res['cnn_arch'].unique()]
    mask = (
        (cnn_res['task'] == task) & (cnn_res['metric'] == metric) &
        (cnn_res['model'] == model) & (cnn_res['analysis_type'] == analysis_type)
    )
    if pair_mode:
        mask &= cnn_res['pair_mode'].fillna('') == pair_mode
    df = cnn_res[mask]
    if df.empty:
        return

    for arch in archs:
        adf = df[df['cnn_arch'] == arch]
        per_seed = adf.groupby(['scaler', 'channels', 'cnn_seed'])['value'].mean().reset_index()
        fig, axes = plt.subplots(1, 3, figsize=figsize, sharey=True)
        for ax, scaler in zip(axes, CNN_SCALER_ORDER):
            sdf = per_seed[per_seed['scaler'] == scaler]
            if sdf.empty:
                ax.set_visible(False)
                continue
            sns.violinplot(
                data=sdf, x='channels', y='value', order=CNN_CHANNEL_ORDER,
                palette=list(CNN_CHANNEL_PALETTE.values()),
                ax=ax, inner='box', linewidth=0.8,
            )
            ax.set_title(f'scaler={scaler}')
            ax.set_xlabel('')
            ax.set_ylabel(metric if ax == axes[0] else '')
            ax.tick_params(axis='x', rotation=30)
        pm_label = f' | {pair_mode}' if pair_mode else ''
        fig.suptitle(f'CNN-seed variability — {task}/{metric}/{model} arch={arch}{pm_label}', fontsize=11)
        fig.tight_layout()
        plt.show()


plot_seed_variability(cnn_res, 'age', 'pearson', 'ridge_reg')
plot_seed_variability(cnn_res, 'age', 'mae', 'ridge_reg')
plot_seed_variability(cnn_res, 'sex', 'auc', 'logistic_clf')
plot_seed_variability(cnn_res, 'delta_age', 'pearson', 'ridge_reg',
                      analysis_type='longitudinal', pair_mode='concat_delta')

## A6. Scaler effect (CNN)

Mean ± std per (channel, scaler) for key metrics.

In [ ]:
def plot_scaler_effect(cnn_summary, task, metric, model, analysis_type='crosssectional',
                       pair_mode=None, figsize=(8, 4)):
    archs = [a for a in CNN_ARCH_ORDER if a in cnn_summary['cnn_arch'].unique()]
    mask = (
        (cnn_summary['task'] == task) & (cnn_summary['metric'] == metric) &
        (cnn_summary['model'] == model) & (cnn_summary['analysis_type'] == analysis_type)
    )
    if pair_mode:
        mask &= cnn_summary['pair_mode'].fillna('') == pair_mode
    df = cnn_summary[mask]
    if df.empty:
        return

    for arch in archs:
        adf = df[df['cnn_arch'] == arch]
        fig, ax = plt.subplots(figsize=figsize)
        for ch in CNN_CHANNEL_ORDER:
            cdf = adf[adf['channels'] == ch].sort_values('scaler')
            if cdf.empty:
                continue
            ax.plot(cdf['scaler'].astype(str), cdf['mean'], marker='o',
                    label=ch, color=CNN_CHANNEL_PALETTE[ch])
            ax.fill_between(cdf['scaler'].astype(str),
                            cdf['mean'] - cdf['std'], cdf['mean'] + cdf['std'],
                            alpha=0.15, color=CNN_CHANNEL_PALETTE[ch])
        pm_label = f' | {pair_mode}' if pair_mode else ''
        ax.set_title(f'{task}/{metric}/{model} arch={arch}{pm_label} — scaler effect (mean ± std)')
        ax.set_xlabel('scaler')
        ax.set_ylabel(metric)
        ax.legend(title='channels', fontsize=8)
        fig.tight_layout()
        plt.show()


plot_scaler_effect(cnn_summary, 'age', 'pearson', 'ridge_reg')
plot_scaler_effect(cnn_summary, 'age', 'mae', 'ridge_reg')
plot_scaler_effect(cnn_summary, 'sex', 'auc', 'logistic_clf')
plot_scaler_effect(cnn_summary, 'delta_age', 'pearson', 'ridge_reg',
                   analysis_type='longitudinal', pair_mode='concat_delta')

---
# Part B — CNN vs FreeSurfer ROI

## B1. Head-to-head: best CNN vs ROI feature sets

For each task, the best CNN configuration (per scaler × channel, best downstream model) is shown alongside each ROI feature set.  
Each bar = mean over all seeds (CNN: 10 seeds × 10 splits; ROI: 1 seed × 10 splits). Error bar = ±1 std.

In [ ]:
if roi_res.empty:
    print('[SKIP] No ROI results. See setup cell for how to generate them.')
else:
    TASK_SPECS = [
        ('age',          'pearson', 'ridge_reg',    'crosssectional', None),
        ('age',          'mae',     'ridge_reg',    'crosssectional', None),
        ('sex',          'auc',     'logistic_clf', 'crosssectional', None),
#        ('session_type', 'balanced_accuracy', 'rf', 'crosssectional', None),
#        ('studies',      'balanced_accuracy', 'rf', 'crosssectional', None),
#        ('study_num',    'balanced_accuracy', 'rf', 'crosssectional', None),
        ('delta_age',    'pearson', 'ridge_reg',    'longitudinal',   'concat_delta'),
        ('delta_age',    'mae',     'ridge_reg',    'longitudinal',   'concat_delta'),
    ]

    def _query_summary(s, task, metric, model, atype, pm, cnn_arch=None):
        mask = (
            (s['task'] == task) & (s['metric'] == metric) &
            (s['model'] == model) & (s['analysis_type'] == atype)
        )
        if pm:
            mask &= s['pair_mode'].fillna('') == pm
        if cnn_arch:
            mask &= s['cnn_arch'] == cnn_arch
        return s[mask]

    def head_to_head(summary, task, metric, model, atype, pm, figsize=(12, 5)):
        archs = [a for a in CNN_ARCH_ORDER if a != 'freesurfer_roi'
                 and a in summary['cnn_arch'].unique()]

        # Build records: one per (label, arch_type)
        records = []

        # CNN: best config per arch (highest/lowest mean)
        for arch in archs:
            sub = _query_summary(summary, task, metric, model, atype, pm, cnn_arch=arch)
            if sub.empty:
                continue
            asc = metric in LOWER_IS_BETTER
            best = sub.sort_values('mean', ascending=asc).iloc[0]
            records.append({
                'label': f"{arch}\nbest ({best['scaler']}/{best['channels']})",
                'mean': best['mean'], 'std': best['std'],
                'arch_type': arch,
            })

        # ROI: all feature sets
        roi_sub = _query_summary(summary, task, metric, model, atype, pm, cnn_arch='freesurfer_roi')
        for _, r in roi_sub.sort_values('channels').iterrows():
            records.append({
                'label': f"ROI\n{r['channels']} ({r['scaler']})",
                'mean': r['mean'], 'std': r['std'],
                'arch_type': 'freesurfer_roi',
            })

        if not records:
            return

        rdf = pd.DataFrame(records)
        colors = [ARCH_PALETTE.get(a, '#888888') for a in rdf['arch_type']]

        fig, ax = plt.subplots(figsize=figsize)
        bars = ax.bar(range(len(rdf)), rdf['mean'], color=colors,
                      yerr=rdf['std'], capsize=4, edgecolor='white', linewidth=0.5)
        ax.set_xticks(range(len(rdf)))
        ax.set_xticklabels(rdf['label'], rotation=30, ha='right', fontsize=8)
        ax.set_ylabel(metric)
        pm_label = f' | {pm}' if pm else ''
        ax.set_title(f'{task} / {metric} / {model}{pm_label}  —  CNN vs ROI (mean ± std)')

        legend_patches = [mpatches.Patch(color=v, label=k) for k, v in ARCH_PALETTE.items()
                          if k in rdf['arch_type'].values]
        ax.legend(handles=legend_patches, fontsize=8)
        fig.tight_layout()
        plt.show()

    for spec in TASK_SPECS:
        task, metric, model, atype, pm = spec
        if task not in summary['task'].unique():
            continue
        head_to_head(summary, task, metric, model, atype, pm)

## B2. ROI feature sets breakdown

Within-ROI comparison across all tasks and scalers.

In [ ]:
if roi_res.empty:
    print('[SKIP] No ROI results.')
else:
    ROI_TASK_SPECS = [
        ('age',          ['pearson', 'mae'],              'ridge_reg',    'crosssectional', None),
        ('sex',          ['auc', 'balanced_accuracy'],    'logistic_clf', 'crosssectional', None),
        ('session_type', ['balanced_accuracy'],           'rf',       'crosssectional', None),
        ('delta_age',    ['pearson', 'mae'],              'ridge_reg',    'longitudinal',   'concat_delta'),
    ]

    for task, metrics, model, atype, pm in ROI_TASK_SPECS:
        mask = (
            (roi_summary['task'] == task) &
            (roi_summary['model'] == model) &
            (roi_summary['analysis_type'] == atype) &
            (roi_summary['metric'].isin(metrics))
        )
        if pm:
            mask &= roi_summary['pair_mode'].fillna('') == pm
        df = roi_summary[mask]
        if df.empty:
            continue

        n_metrics = len(metrics)
        n_scalers = df['scaler'].nunique()
        fig, axes = plt.subplots(n_metrics, n_scalers,
                                  figsize=(5 * n_scalers, 4 * n_metrics),
                                  sharey='row', squeeze=False)

        for row, metric in enumerate(metrics):
            mdf = df[df['metric'] == metric]
            scalers = sorted(mdf['scaler'].dropna().unique())
            for col, scaler in enumerate(scalers):
                ax = axes[row, col]
                sdf = mdf[mdf['scaler'] == scaler].dropna(subset=['channels']).sort_values('channels')
                ch_labels = list(sdf['channels'].astype(str))
                colors = [ROI_CHANNEL_PALETTE.get(c, '#888') for c in ch_labels]
                ax.bar(ch_labels, sdf['mean'],
                       yerr=sdf['std'], color=colors, capsize=4,
                       edgecolor='white', linewidth=0.5)
                ax.set_title(f'scaler={scaler}' if row == 0 else '')
                ax.set_ylabel(metric if col == 0 else '')
                ax.tick_params(axis='x', rotation=30)
                ax.set_xlabel('')

        pm_label = f' | {pm}' if pm else ''
        fig.suptitle(f'ROI feature sets — {task} / {model}{pm_label}', fontsize=12, y=1.01)
        fig.tight_layout()
        plt.show()

## B3. Full heatmap — CNN + ROI combined

All architectures in one view. Columns sorted: double_conv configs, then cov_pool configs, then ROI feature sets.

In [ ]:
if roi_res.empty:
    print('[SKIP] No ROI results — showing CNN-only heatmap.')
    plot_heatmap(cnn_summary)
else:
    plot_heatmap(summary, title='Full heatmap — CNN + FreeSurfer ROI (row-normalized, green=best)',
                 cell_fontsize=5)

## B4. Summary tables — top configurations per task (all sources)

In [ ]:
TOP_N = 10
TASK_SPECS_TABLE = [
    ('age',          'pearson', 'ridge_reg',    'crosssectional', None),
    ('sex',          'auc',     'logistic_clf', 'crosssectional', None),
    ('session_type', 'balanced_accuracy', 'rf', 'crosssectional', None),
    ('studies',      'balanced_accuracy', 'rf', 'crosssectional', None),
    ('study_num',    'balanced_accuracy', 'rf', 'crosssectional', None),
    ('delta_age',    'pearson', 'ridge_reg',    'longitudinal',   'concat_delta'),
    ('delta_age',    'mae',     'ridge_reg',    'longitudinal',   'concat_delta'),
]

for task, metric, model, atype, pm in TASK_SPECS_TABLE:
    mask = (
        (summary['task'] == task) & (summary['metric'] == metric) &
        (summary['model'] == model) & (summary['analysis_type'] == atype)
    )
    if pm:
        mask &= summary['pair_mode'].fillna('') == pm
    sub = summary[mask]
    if sub.empty:
        continue
    asc = metric in LOWER_IS_BETTER
    top = (
        sub[['cnn_arch', 'scaler', 'channels', 'mean', 'std', 'n']]
        .sort_values('mean', ascending=asc)
        .head(TOP_N)
    )
    pm_label = f' | {pm}' if pm else ''
    print(f"\n=== {task} | {metric} | {model} | {atype}{pm_label} ===")
    print(top.to_string(index=False))

## B5. Cross-task correlation — per configuration

Does a configuration that predicts age well also predict sex well? Includes both CNN and ROI configs.

In [ ]:
CROSS_TASK = [
    ('age',          'pearson', 'ridge_reg',    'crosssectional', None),
    ('sex',          'auc',     'logistic_clf', 'crosssectional', None),
    ('session_type', 'balanced_accuracy', 'rf', 'crosssectional', None),
    ('studies',      'balanced_accuracy', 'rf', 'crosssectional', None),
    ('delta_age',    'pearson', 'ridge_reg',    'longitudinal',   'concat_delta'),
]

frames = []
for task, metric, model, atype, pm in CROSS_TASK:
    mask = (
        (summary['task'] == task) & (summary['metric'] == metric) &
        (summary['model'] == model) & (summary['analysis_type'] == atype)
    )
    if pm:
        mask &= summary['pair_mode'].fillna('') == pm
    sub = summary[mask][['cnn_arch', 'scaler', 'channels', 'mean']].copy()
    sub = sub.rename(columns={'mean': f'{task}/{metric}'})
    frames.append(sub.set_index(['cnn_arch', 'scaler', 'channels']))

if frames:
    wide = pd.concat(frames, axis=1).dropna(how='all')
    corr = wide.corr()

    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(
        corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
        vmin=-1, vmax=1, ax=ax, square=True, linewidths=0.5,
        mask=np.triu(np.ones_like(corr, dtype=bool), k=1),
    )
    ax.set_title('Cross-task correlation — all configurations (CNN + ROI)', pad=10)
    fig.tight_layout()
    plt.show()